# Label-free KNN demo similarity audit
Compare `unknown_demo_kb_updates.parquet` against the frozen initial KB using numerical flow features only. Save KB-matched and AE-routed records separately. No BENIGN records, label-based matching, AE scoring, or KB updates.


In [1]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
import joblib
ROOT=Path.cwd().resolve()
if ROOT.name.lower()=="kb" and ROOT.parent.name.lower()=="notebooks": ROOT=ROOT.parent.parent
elif ROOT.name.lower()=="notebooks": ROOT=ROOT.parent
KB_DIR=ROOT/"notebooks"/"kb"
if str(KB_DIR) not in sys.path: sys.path.insert(0,str(KB_DIR))
from kb_flow import read_partition, search, EXCLUDE
SPLITS=ROOT/"data/processed/Data_V3_SSH_Unknown_final/unknown_kb_splits"
DATA=ROOT/"data/processed/Data_V3_SSH_Unknown_final"
OUT=ROOT/"data/processed/knn_kb/demo_similarity_audit"
KB_CSV=ROOT/"data/processed/knn_kb/knowledge_base.csv"
INDEX=ROOT/"saved-models/knn_kb/initial_kb.joblib"
for p in [SPLITS/"unknown_initial_kb.parquet",SPLITS/"unknown_demo_kb_updates.parquet",KB_CSV,INDEX]:
    if not p.is_file(): raise FileNotFoundError(p)
OUT.mkdir(parents=True,exist_ok=True)
reference=pd.read_csv(KB_CSV,low_memory=False)
model=joblib.load(INDEX)  # Only load your own trusted artifacts.
if len(reference)!=model['count']: raise ValueError("Frozen KB CSV/index count mismatch")
if model.get('cutoff') is None: raise ValueError("Run kb_training.ipynb first to calibrate the KNN cutoff")
forbidden = [c for c in model['columns'] if c.strip().casefold() in EXCLUDE]
if forbidden:
    raise ValueError(f"Excluded metadata found in KNN feature list: {forbidden}")
initial=read_partition(SPLITS/"unknown_initial_kb.parquet")
if len(initial)!=len(reference): raise ValueError("Initial Parquet and KB CSV counts differ")
cols=model['columns']
if not np.allclose(initial[cols].apply(pd.to_numeric,errors='coerce').to_numpy(dtype=float),
                   reference[cols].apply(pd.to_numeric,errors='coerce').to_numpy(dtype=float),
                   equal_nan=True,rtol=1e-6,atol=1e-8):
    raise ValueError("Initial Parquet and KB CSV record ordering differs")
print("Unknown attack demo source:",SPLITS/"unknown_demo_kb_updates.parquet")
print("Demo outputs:",OUT)


Unknown attack demo source: C:\Users\harsh\Downloads\Projects\Hybrid-Network-Intrusion-Detection\data\processed\Data_V3_SSH_Unknown_final\unknown_kb_splits\unknown_demo_kb_updates.parquet
Demo outputs: C:\Users\harsh\Downloads\Projects\Hybrid-Network-Intrusion-Detection\data\processed\knn_kb\demo_similarity_audit


In [2]:
# Compare the separate unknown demo flows against the FROZEN initial KB.
demo=read_partition(SPLITS/"unknown_demo_kb_updates.parquet")
attack_hits=search(demo,model,reference,k=5)
attack_audit=pd.DataFrame([{
    "demo_row":i,"matched":bool(h['matched']),
    "nearest_distance":float(h['nearest_distance']),
    "nearest_pattern_id":h['neighbors'][0]['pattern_id'],
    "route":"KB_MATCH" if h['matched'] else "AUTOENCODER",
} for i,h in enumerate(attack_hits)])
# Keep all original flow fields except excluded label/attempt metadata in demo exports.
# The index is built from model['columns'] only; no label ever enters KNN.
metadata_to_drop = [c for c in demo.columns if c.strip().casefold() in
                    {'label', 'attempted category', 'source_file', 'is_attempted', 'id'}]
attack_tagged=demo.drop(columns=metadata_to_drop).copy()
attack_tagged.insert(0,"demo_row",np.arange(len(demo)))
mask=attack_audit['matched'].to_numpy(dtype=bool)
attack_tagged.loc[mask].to_parquet(OUT/"kb_demo_matched.parquet",index=False)
attack_tagged.loc[~mask].to_parquet(OUT/"autoencoder_demo_unmatched.parquet",index=False)
attack_audit.to_csv(OUT/"demo_similarity_audit.csv",index=False)
print("KNN numerical features:", len(model['columns']))
print("Match cutoff:",model['cutoff'])
print("ATTACK DEMO: KB matches:",int(mask.sum()),"AE candidates:",int((~mask).sum()))
display(attack_audit.head(10))

attack_audit.loc[mask, ["demo_row", "nearest_distance", "nearest_pattern_id"]].to_csv(OUT/"kb_demo_row_ids.csv",index=False)
attack_audit.loc[~mask, ["demo_row", "nearest_distance", "nearest_pattern_id"]].to_csv(OUT/"autoencoder_demo_row_ids.csv",index=False)
print("Saved KB and AE attack demo partitions under:", OUT)


KNN numerical features: 73
Match cutoff: 0.9453825241925351
ATTACK DEMO: KB matches: 420 AE candidates: 14


,demo_row,matched,nearest_distance,nearest_pattern_id,route
0,0,True,0.047224,KB_0000086,KB_MATCH
1,1,True,0.072170,KB_0001464,KB_MATCH
2,2,True,0.113858,KB_0001326,KB_MATCH
3,3,True,0.077619,KB_0001901,KB_MATCH
4,4,True,0.105062,KB_0000309,KB_MATCH
5,5,True,0.285570,KB_0002541,KB_MATCH
6,6,True,0.374752,KB_0004624,KB_MATCH
7,7,True,0.124801,KB_0002190,KB_MATCH
8,8,True,0.031651,KB_0000122,KB_MATCH
9,9,True,0.128913,KB_0001168,KB_MATCH


Saved KB and AE attack demo partitions under: C:\Users\harsh\Downloads\Projects\Hybrid-Network-Intrusion-Detection\data\processed\knn_kb\demo_similarity_audit
